In [5]:
# =============================================================================
# Fig5: Quadrant plots — Δ Bed Occupancy Rate, BEFORE-mobility vs AFTER-mobility,
#       for three periods: 2020→2040, 2040→2060, 2020→2060.
#
# For each city and each period:
#   x = Δ Occ_before = Occ_before(year_end) - Occ_before(year_start)
#       "how pressure would have evolved with NO mobility" (counterfactual)
#   y = Δ Occ_after  = Occ_after(year_end)  - Occ_after(year_start)
#       "how pressure actually evolved, WITH mobility"
#
# Quadrant interpretation (reference lines at x=0, y=0):
#   bottom-right (x>0, y<0): pressure would have risen, but mobility reversed it
#   bottom-left  (x<0, y<0): pressure was already falling, mobility reinforced it
#   top-right    (x>0, y>0): pressure rose regardless of mobility
#   top-left     (x<0, y>0): pressure was falling, but mobility reversed it upward
#
# This reuses the same per-disease occupancy pipeline as Fig4, but WITHOUT
# averaging across years — each of 2020/2040/2060 is computed as its own
# single-year snapshot.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ── 0. Paths & settings (same sources as Fig4) ────────────────────────────────
CITYSUM_BYDISEASE_DIR = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease")
CITYSUM_BYDISEASE_TPL = "citysum_bydisease_{air}_{year}.csv"

DISEASE_FLOW_DIR = Path("/Volumes/UCL/论文工作/空气污染/cross_flow_by_disease/flow_results")
DISEASE_FLOW_TPL = "flow_patientnum_{disease_tag}_{air}_{year}.csv"

INPUT_FILE = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/air_pollution/data source/hospital/13-National hospital directory.xlsx")
INCOME_DIR       = Path("/Volumes/UCL/论文工作/空气污染/weighted_gdp/avg_fer_rcp/avg_2020.csv")
INCOME_CITY_COL  = "city"
INCOME_CLASS_COL = "income_group"

SHP_PATH   = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")

OUTFILE = Path("/Users/shirley/Desktop/plots_V2/Fig5_quadrant_occupancy_change.png")
OUTFILE.parent.mkdir(parents=True, exist_ok=True)

SCENARIO = "earlypeak_NZ_CL"

GDP_CLASS_ORDER  = ["Low", "Lower middle", "Middle", "Upper middle", "High"]
GDP_CLASS_SHORT  = {"Low": "Low", "Lower middle": "L-Mid", "Middle": "Mid",
                     "Upper middle": "U-Mid", "High": "High"}
GDP_CLASS_COLORS = {"Low": "#D6EFD6", "Lower middle": "#A8DDA8", "Middle": "#6FC26F",
                     "Upper middle": "#3B9E3B", "High": "#1B6B1B"}

# Three periods to plot, each as (year_start, year_end, panel_title)
PERIODS = [
    (2020, 2040, "2020\u20132040"),
    (2040, 2060, "2040\u20132060"),
    (2020, 2060, "2020\u20132060"),
]

plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        7,
    "axes.titlesize":   8,
    "axes.titleweight": "bold",
    "axes.titlepad":    3,
})

# ── 0b. Length-of-stay lookup (Table S3) — identical to Fig4 ─────────────────
LOS_BY_SHORT_NAME = {
    "COPD": 10.0, "IHD": 12.6, "LRI": 11.0, "LC": 14.2, "STROKE": 19.7,
}

def match_los(disease_name: str) -> float:
    s = disease_name.strip().lower()
    if "copd" in s or "chronic obstructive" in s:
        return LOS_BY_SHORT_NAME["COPD"]
    if "isch" in s or "ihd" in s or "coronary" in s or "chd" in s:
        return LOS_BY_SHORT_NAME["IHD"]
    if "lower respiratory" in s or s == "lri" or "lri" in s:
        return LOS_BY_SHORT_NAME["LRI"]
    if "lung cancer" in s or s == "lc":
        return LOS_BY_SHORT_NAME["LC"]
    if "stroke" in s:
        return LOS_BY_SHORT_NAME["STROKE"]
    raise ValueError(f"Could not match disease name '{disease_name}' to a known LOS category.")

CITY_NAME_MAP = {
    "Wulumuqi": "Urumqi", "Xian": "Xi'an", "Qiqihaer": "Qiqihar",
    "Huhehaote": "Hohhot", "Haerbin": "Harbin", "Xiuqian": "Suqian",
    "Wulanchabu": "Ulanqab", "Shaoang": "Zhaotong", "Tongcuan": "Tongchuan",
    "Xiangfan": "Xiangyang", "Akesu": "Aksu", "Alaer": "Alar",
    "Alashan": "Alxa", "Bayanzhuoer": "Bayannur", "Bayinguoleng": "Bayingolin",
    "Boertalameng": "Bortala", "Changdu": "Qamdo", "Eerduosi": "Erdos",
    "Ordos": "Erdos", "Guoluo": "Golog", "Guolo": "Golog",
    "Hulunbeier": "Hulunbuir", "Kezilesu": "Kizilsu", "Ledong": "Ledong",
    "Lingshui": "Lingshui", "Linzhi": "Nyingchi", "Naqu": "Nagqu",
    "Qiongzhong": "Qiongzhong", "Shennongjia": "Shennongjia",
    "Tumushuke": "Tumxuk", "Xilinguole": "Xilingol", "Xingan": "Hinggan",
}

def disease_to_tag(disease):
    return disease.strip().replace(" ", "_").replace("/", "-")

# ── 1. Per-disease occupancy loaders (single-year, no averaging) ─────────────
def load_local_bydisease(year):
    path = CITYSUM_BYDISEASE_DIR / CITYSUM_BYDISEASE_TPL.format(air=SCENARIO, year=year)
    df = pd.read_csv(path)
    df["city"]    = df["city"].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    df["disease"] = df["disease"].astype(str).str.strip()
    return df

def load_disease_flow_matrix(year, disease_tag):
    path = DISEASE_FLOW_DIR / DISEASE_FLOW_TPL.format(disease_tag=disease_tag, air=SCENARIO, year=year)
    if not path.exists():
        return None
    df = pd.read_csv(path, index_col=0)
    df.index   = df.index.str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    df.columns = df.columns.str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    df = df.loc[~df.index.isin(["total"]), ~df.columns.isin(["total"])]
    return df

def compute_year_occupancy_days(year):
    """Same logic as Fig4: occ_before uses mo_total (no-mobility counterfactual),
    occ_after uses local_mo_total + inflow (actual/simulated mobility)."""
    local_df = load_local_bydisease(year)
    diseases = sorted(local_df["disease"].unique().tolist())

    before_acc, after_acc = None, None
    for disease in diseases:
        los = match_los(disease)
        disease_tag = disease_to_tag(disease)

        sub = local_df[local_df["disease"] == disease].groupby("city").sum(numeric_only=True)
        mo_total_d = sub["mo_total_sum"]
        local_mo_d = sub["local_mo_total"]

        flow_mat = load_disease_flow_matrix(year, disease_tag)
        if flow_mat is None:
            inflow_d = pd.Series(0.0, index=mo_total_d.index)
        else:
            inflow_d = flow_mat.sum(axis=0).groupby(level=0).sum()

        all_cities = mo_total_d.index.union(local_mo_d.index).union(inflow_d.index)
        mo_total_d = mo_total_d.reindex(all_cities, fill_value=0.0)
        local_mo_d = local_mo_d.reindex(all_cities, fill_value=0.0)
        inflow_d   = inflow_d.reindex(all_cities, fill_value=0.0)

        demand_after_d = (local_mo_d + inflow_d).clip(lower=0.0)

        before_days_d = mo_total_d * los
        after_days_d  = demand_after_d * los

        before_acc = before_days_d if before_acc is None else before_acc.add(before_days_d, fill_value=0.0)
        after_acc  = after_days_d  if after_acc  is None else after_acc.add(after_days_d,  fill_value=0.0)

    return pd.DataFrame({"occ_before_days": before_acc, "occ_after_days": after_acc})

# ── 2. Resource & income-class loaders ────────────────────────────────────────
def load_resource():
    df = pd.read_excel(INPUT_FILE, sheet_name="fig4")
    df = df[["city", "beds", "clinics"]].copy()
    df["beds"]    = pd.to_numeric(df["beds"],    errors="coerce")
    df["clinics"] = pd.to_numeric(df["clinics"], errors="coerce")
    df = df.dropna(subset=["city", "beds", "clinics"])
    df = df[(df["beds"] > 0) & (df["clinics"] > 0)]
    df["city"] = df["city"].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    return df.groupby("city")["beds"].sum()

def load_gdp_class():
    df = pd.read_csv(INCOME_DIR)
    df[INCOME_CITY_COL] = df[INCOME_CITY_COL].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    df = df.dropna(subset=[INCOME_CITY_COL, INCOME_CLASS_COL])
    df["gdp_class"] = pd.Categorical(df[INCOME_CLASS_COL], categories=GDP_CLASS_ORDER, ordered=True)
    return df.set_index(INCOME_CITY_COL)["gdp_class"]

# region (East/West) via Hu Line, reusing the same shapefile-based method as Fig4
import geopandas as gpd
from pyproj import Transformer

PROJ_STR = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105"

def load_region_map():
    city_shp_raw = gpd.read_file(SHP_PATH)
    city_shp_raw["English"] = city_shp_raw["English"].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    city_shp = city_shp_raw.to_crs(PROJ_STR)

    _tf = Transformer.from_crs("EPSG:4326", PROJ_STR, always_xy=True)
    HHY_X, HHY_Y = _tf.transform([127.5, 98.5], [50.2, 25.0])
    x0, y0, x1, y1 = HHY_X[0], HHY_Y[0], HHY_X[1], HHY_Y[1]

    def _hhy_x_at_y(y):
        t = (y - y1) / (y0 - y1)
        return x1 + t * (x0 - x1)

    cx = city_shp.geometry.centroid.x
    cy = city_shp.geometry.centroid.y
    city_shp["region"] = np.where(cx > _hhy_x_at_y(cy), "East", "West")
    return (city_shp[["English", "region"]]
            .drop_duplicates(subset="English")
            .set_index("English")["region"].to_dict())

# ── 3. Compute occupancy (%) for each of the needed years ────────────────────
resource  = load_resource()
gdp_class = load_gdp_class()
region_map = load_region_map()

needed_years = sorted({y for pair in PERIODS for y in (pair[0], pair[1])})
occ_by_year = {}
for year in needed_years:
    occ_days = compute_year_occupancy_days(year)
    common = occ_days.index.intersection(resource.index)
    occ_days = occ_days.loc[common]
    res = resource.loc[common]
    occ_by_year[year] = pd.DataFrame({
        "Occ_before": occ_days["occ_before_days"] / (res * 365) * 100,
        "Occ_after":  occ_days["occ_after_days"]  / (res * 365) * 100,
    })
    print(f"✓ Computed occupancy for year {year} | {len(occ_by_year[year])} cities")

# ── 4. Build the Δbefore vs Δafter table for each period ─────────────────────
period_data = {}
for y0, y1, label in PERIODS:
    df0, df1 = occ_by_year[y0], occ_by_year[y1]
    common = df0.index.intersection(df1.index)
    d = pd.DataFrame({
        "d_before": df1.loc[common, "Occ_before"] - df0.loc[common, "Occ_before"],
        "d_after":  df1.loc[common, "Occ_after"]  - df0.loc[common, "Occ_after"],
    })
    d["gdp_class"] = d.index.map(gdp_class)
    d["region"]    = d.index.map(region_map)
    d = d.dropna(subset=["gdp_class"])
    period_data[label] = d

# ── 5. Draw the three quadrant panels ─────────────────────────────────────────
fig = plt.figure(figsize=(180 / 25.4, 65 / 25.4), dpi=300)
gs = GridSpec(1, 3, figure=fig, wspace=0.30, left=0.07, right=0.98, top=0.86, bottom=0.16)

for j, (label, d) in enumerate(period_data.items()):
    ax = fig.add_subplot(gs[0, j])

    # x = Δ Occ_after (actual, with mobility)
    # y = Δ Occ_before (counterfactual, no mobility)
    #
    # Quadrant meanings under this orientation:
    #   top-left    (x<0, y>0): RELIEF — pressure would have risen, mobility
    #                            reversed it into an actual decline
    #   bottom-right(x>0, y<0): DETERIORATION — pressure would have fallen,
    #                            but mobility pushed it into an actual rise
    #   top-right   (x>0, y>0): pressure rose regardless of mobility
    #   bottom-left (x<0, y<0): pressure was already falling, reinforced

    # NEW: per-panel axis range, based on THIS panel's own distribution
    # (robust to a single outlier city stretching every panel to the same
    # wide range). Uses the 99th percentile of |value| with a small margin,
    # then clips to the true max so the range never falls short of any point.
    panel_vals = d[["d_before", "d_after"]].values.flatten()
    panel_vals = panel_vals[np.isfinite(panel_vals)]
    p99 = np.nanpercentile(np.abs(panel_vals), 99) if len(panel_vals) else 1
    true_max = np.nanmax(np.abs(panel_vals)) if len(panel_vals) else 1
    lim = max(p99 * 1.25, true_max * 1.02)  # small margin, but never clip actual points

    ax.axhline(0, color="#999999", lw=0.7, zorder=1)
    ax.axvline(0, color="#999999", lw=0.7, zorder=1)

    # quadrant shading: green = relief (top-left), red = deterioration (bottom-right)
    ax.axhspan(0, lim, xmin=0.5, xmax=1.0, color="#f5f5f5", alpha=0.5, zorder=0)   # top-right: rose regardless
    ax.axhspan(-lim, 0, xmin=0.5, xmax=1.0, color="#fdecea", alpha=0.6, zorder=0)  # bottom-right: DETERIORATION
    ax.axhspan(-lim, 0, xmin=0.0, xmax=0.5, color="#f5f5f5", alpha=0.5, zorder=0)  # bottom-left: falling, reinforced
    ax.axhspan(0, lim, xmin=0.0, xmax=0.5, color="#eaf5ea", alpha=0.6, zorder=0)   # top-left: RELIEF

    for cls in GDP_CLASS_ORDER:
        sub = d[d["gdp_class"] == cls]
        ax.scatter(sub["d_after"], sub["d_before"], s=10, color=GDP_CLASS_COLORS[cls],
                   edgecolors="none", alpha=0.85, zorder=3, label=GDP_CLASS_SHORT[cls])

    ax.plot([-lim, lim], [-lim, lim], color="#888888", lw=0.6, ls=(0, (3, 2)), zorder=2)

    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    # Force consistent tick formatting across all three panels (integer ticks,
    # similar tick count) — otherwise matplotlib's default locator can pick
    # decimal steps (e.g. 2.5) for one panel's data range and whole-number
    # steps for another, making the panels look inconsistently labeled.
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=6, integer=True, symmetric=True))
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=6, integer=True, symmetric=True))
    ax.set_xlabel("Δ Occupancy, after mobility (pp)", fontsize=6.5)
    if j == 0:
        ax.set_ylabel("Δ Occupancy, before mobility (pp)", fontsize=6.5)
    ax.tick_params(labelsize=6)
    ax.set_title(label, fontsize=8, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

    # Count and report ALL FOUR quadrants (not just relief)
    n = len(d)
    q_relief   = ((d["d_after"] < 0) & (d["d_before"] > 0)).sum()  # top-left
    q_worsen   = ((d["d_after"] > 0) & (d["d_before"] < 0)).sum()  # bottom-right
    q_rose     = ((d["d_after"] > 0) & (d["d_before"] > 0)).sum()  # top-right
    q_fell     = ((d["d_after"] < 0) & (d["d_before"] < 0)).sum()  # bottom-left

    def _pct(x):
        return f"{x/n*100:.0f}%" if n else "0%"

    summary_text = (
        f"n={n}\n"
        f"relief: {q_relief} ({_pct(q_relief)})\n"
        f"worsened: {q_worsen} ({_pct(q_worsen)})\n"
        f"rose regardless: {q_rose} ({_pct(q_rose)})\n"
        f"fell regardless: {q_fell} ({_pct(q_fell)})"
    )
    ax.text(0.97, 0.03, summary_text,
            transform=ax.transAxes, fontsize=5, color="#333333", ha="right", va="bottom",
            linespacing=1.4)

fig.legend(
    handles=[plt.Line2D([0], [0], marker="o", color="w",
                         markerfacecolor=GDP_CLASS_COLORS[c], markersize=5,
                         label=GDP_CLASS_SHORT[c]) for c in GDP_CLASS_ORDER],
    fontsize=6, frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.0)
)

fig.savefig(OUTFILE, dpi=400, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"\n✓ Saved → {OUTFILE}")

✓ Computed occupancy for year 2020 | 350 cities
✓ Computed occupancy for year 2040 | 350 cities
✓ Computed occupancy for year 2060 | 350 cities

✓ Saved → /Users/shirley/Desktop/plots_V2/Fig5_quadrant_occupancy_change.png
